# Fitness estimation

This is an exploration to estimate the fitness of a mutation from a single timepoint.

Just run a series of simulations at different fitnesses for a given induction and then read off the end takeover

In [1]:
from pyabc import (ABCSMC, RV, Distribution, PNormDistance)
from pyabc.sampler import MulticoreEvalParallelSampler, SingleCoreSampler
from functools import partial, update_wrapper
import pandas as pd
import numpy as np
from scipy.stats import ks_2samp
from collections import OrderedDict
from clone_competition_simulation.parameters import Parameters
import pyabc.visualization.credible as credible
import sys

from clone_competition_simulation.parameters import Parameters, TimeParameters, PopulationParameters, LabelParameters, FitnessParameters

# Functions and fixed parameters for simulations
GRID_SHAPE = (500, 500)
DIVISION_RATE = 0.27
CELLS = GRID_SHAPE[0]*GRID_SHAPE[1]

ERROR_OBJECT = {'distance': np.inf}

INDUCTION = 0.002

#NUM_CLONES = 100

# Each 500x500 grid is 0.25 million cells.
# If 1million cells in the oesophagus, then 4 grids is a full oesophagus.
# Asking for 100 clones in total for each time point.
# Run simulations repeatedly until enough clones exist for each time point.
# Don't want to get stuck in a loop if the induction/fitness is too low and no clones survive
# Limit the number of loops. If 4 grids per oeosophagus, then 50 grids is 12.5 mice worth, more than sampled number.
LOOP_LIMITS = 50


def get_grid(fitness, induction, grid_shape, cells):
    # Make the initial grid with randomly placed induced cells
    initial_grid = np.zeros(grid_shape, dtype=int)
    total_mutants = int(induction*cells)
    mutant_locs = np.random.choice(grid_shape[0]*grid_shape[1], total_mutants, replace=False)
    mutant_locs = [(m // grid_shape[1], m % grid_shape[1]) for m in mutant_locs]  # Convert the random draws into array indices

    count = 0
    for i in range(total_mutants):
        initial_grid[mutant_locs[count]] = i + 1
        count += 1

    fitness_array = [1] + [fitness]*total_mutants
    label_array = [0] + [1]*total_mutants
    return initial_grid, fitness_array, label_array


In [2]:
def get_mutant_takeover(sim):
    mutants = sim.population_array.toarray()[1:]
    mutant_pop = mutants.sum(axis=0)
    if mutant_pop.max() == sim.total_pop:
        mutant_pop[np.argmax(mutant_pop):] = sim.total_pop
    return mutant_pop/CELLS
def transformBySampling(takeover,samples):
    return([t for (t,individualSample) in zip(takeover,samples) for i in range(individualSample) ])

In [3]:
def run_sim(parameters, times, samplesPerTimepoint, target_data, return_takeover=False):
    #print(parameters)
    fitness, induction = parameters['fitness'], parameters['induction']
    #times = [t for t in target_data]
    #samplesPerTimepoint = [4,4,3,4,4,3]

    try:
        #setup simulation parameters
        initial_grid, fitness_array, label_array = get_grid(fitness, induction, GRID_SHAPE, CELLS)
        if len(fitness_array) == 1:  # Induction rate too low. No mutants on grid.
            return ERROR_OBJECT


        p = Parameters(algorithm='WF2D', 
               population=PopulationParameters(initial_grid=initial_grid,cell_in_own_neighbourhood=True),
               times=TimeParameters(times=times, division_rate=DIVISION_RATE), 
               labels=LabelParameters(initial_label_array=label_array),
               fitness=FitnessParameters(initial_fitness_array=fitness_array),
                       show_progress=False
               )
        '''
        p = Parameters(algorithm='WF2D', initial_grid=initial_grid, times=times, fitness_array=fitness_array,
                       label_array=label_array,
                       print_warnings=False, division_rate=DIVISION_RATE,
                       cell_in_own_neighbourhood=True)'''
        
        #clone_counts = np.zeros(len(times), dtype=int)
        takeover = np.zeros(len(times), dtype=int)
        sim_distributions = [[] for i in range(len(times))]
        if p:
            for loop in range(LOOP_LIMITS):
                s = p.get_simulator()
                s.run_sim()
                takeover = get_mutant_takeover(s)

            if return_takeover:
                return transformBySampling(takeover,samplesPerTimepoint) #takeover
            
            #transform takeover to experimental timepoints
            result = transformBySampling(takeover,samplesPerTimepoint)
            
            #total_distance = sum([distance_ks(target_data[t], res[t]) for t in target_data])
            total_distance = distance_ir2(target_data, result)
            return {'distance': total_distance}
        else:
            return ERROR_OBJECT
    except (Exception, SystemExit) as e:
        print('Error')
        print(e)
        return ERROR_OBJECT

In [4]:
testP = {'fitness':1.01,'induction':INDUCTION}
times = [546]
timepointsPerSample = [1]
targetData = []
goal = 0.08

In [5]:
for i in range(10):
    fitness= 1.0 + i * 0.02
    p = {'fitness':fitness,'induction':INDUCTION}
    print(fitness, run_sim(p,times,timepointsPerSample,targetData,True))

21:19:13 | DEBUG | Times rounded to match simulation steps:  Times used: [544.44444444]
21:22:14 | DEBUG | Times rounded to match simulation steps:  Times used: [544.44444444]


1.0 [np.float64(0.002932)]


21:24:53 | DEBUG | Times rounded to match simulation steps:  Times used: [544.44444444]


1.02 [np.float64(0.008428)]


21:27:30 | DEBUG | Times rounded to match simulation steps:  Times used: [544.44444444]


...

1.12 [np.float64(0.409832)]


21:49:13 | DEBUG | Times rounded to match simulation steps:  Times used: [544.44444444]


1.1400000000000001 [np.float64(0.5549)]


21:51:37 | DEBUG | Times rounded to match simulation steps:  Times used: [544.44444444]


1.16 [np.float64(0.640468)]
1.18 [np.float64(0.787336)]
